<div align="center">

# LOSO Model

</div>

In [10]:
from pathlib import Path
import h5py
import numpy as np

# 1. Load Files

In [73]:
# Seizure files
feature_dir = Path("Features/Seizure")

files = sorted(feature_dir.glob("chb01_*_seizure.mat"))

print(len(files))
for f in files:
    print(f.name)

7
chb01_03_seizure.mat
chb01_04_seizure.mat
chb01_15_seizure.mat
chb01_16_seizure.mat
chb01_18_seizure.mat
chb01_21_seizure.mat
chb01_26_seizure.mat


In [15]:
# Non-Seizure files
feature_dir = Path("Features/Non_Seizure")

files = sorted(feature_dir.glob("chb01_*_features.mat"))

print(len(files))
for f in files:
    print(f.name)

30
chb01_01_features.mat
chb01_02_features.mat
chb01_05_features.mat
chb01_06_features.mat
chb01_07_features.mat
chb01_08_features.mat
chb01_09_features.mat
chb01_10_features.mat
chb01_11_features.mat
chb01_12_features.mat
chb01_13_features.mat
chb01_14_features.mat
chb01_17_features.mat
chb01_19_features.mat
chb01_20_features.mat
chb01_22_features.mat
chb01_23_features.mat
chb01_24_features.mat
chb01_25_features.mat
chb01_27_features.mat
chb01_29_features.mat
chb01_30_features.mat
chb01_31_features.mat
chb01_32_features.mat
chb01_33_features.mat
chb01_34_features.mat
chb01_36_features.mat
chb01_37_features.mat
chb01_38_features.mat
chb01_39_features.mat


In [10]:
# Inspeact one file
with h5py.File(files[0], "r") as f:
    print(list(f.keys()))
    print(f["all_feats"].shape)
    print(f["all_labels"].shape)

IndexError: list index out of range

In [25]:
from pathlib import Path
import numpy as np
import h5py

records = {}

# =====================================================
# LOAD SEIZURE FILES
# =====================================================

seizure_files = sorted(
    Path("Features/Seizure").glob("chb01_*_seizure.mat")
)

for file in seizure_files:

    with h5py.File(file, "r") as f:
        X = f["all_feats"][:]
        y = f["all_labels"][:].squeeze()

    if X.shape[0] == 1530:
        X = X.T

    record_name = file.stem.replace("_seizure", "")

    records[record_name] = (X, y)

    print(
        "[SEIZURE]",
        record_name,
        X.shape,
        y.shape,
        "positives =", int(np.sum(y))
    )

# =====================================================
# LOAD NON-SEIZURE FILES
# =====================================================

non_files = sorted(
    Path("Features/Non_Seizure").glob("chb01_*_features.mat")
)

for file in non_files:

    with h5py.File(file, "r") as f:
        X = f["all_feats"][:]

    if X.shape[0] == 1530:
        X = X.T

    y = np.zeros(X.shape[0], dtype=int)

    record_name = file.stem.replace("_features", "")

    records[record_name] = (X, y)

    print(
        "[NON-SEIZURE]",
        record_name,
        X.shape,
        y.shape,
        "positives =", int(np.sum(y))
    )

print("\nTotal records:", len(records))

[SEIZURE] chb01_03 (1437, 1530) (1437,) positives = 16
[SEIZURE] chb01_04 (1437, 1530) (1437,) positives = 11
[SEIZURE] chb01_15 (1437, 1530) (1437,) positives = 16
[SEIZURE] chb01_16 (1437, 1530) (1437,) positives = 21
[SEIZURE] chb01_18 (1437, 1530) (1437,) positives = 37
[SEIZURE] chb01_21 (1437, 1530) (1437,) positives = 38
[SEIZURE] chb01_26 (927, 1530) (927,) positives = 41
[NON-SEIZURE] chb01_01 (1437, 1530) (1437,) positives = 0
[NON-SEIZURE] chb01_02 (1437, 1530) (1437,) positives = 0
[NON-SEIZURE] chb01_05 (1437, 1530) (1437,) positives = 0
[NON-SEIZURE] chb01_06 (1437, 1530) (1437,) positives = 0
[NON-SEIZURE] chb01_07 (1437, 1530) (1437,) positives = 0
[NON-SEIZURE] chb01_08 (1437, 1530) (1437,) positives = 0
[NON-SEIZURE] chb01_09 (1437, 1530) (1437,) positives = 0
[NON-SEIZURE] chb01_10 (1437, 1530) (1437,) positives = 0
[NON-SEIZURE] chb01_11 (1437, 1530) (1437,) positives = 0
[NON-SEIZURE] chb01_12 (1437, 1530) (1437,) positives = 0
[NON-SEIZURE] chb01_13 (1437, 1530) (

# Code

In [ ]:
from dataclasses import dataclass
import numpy as np
import pandas as pd

from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import GroupKFold
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    average_precision_score,
    recall_score,
    precision_score,
    confusion_matrix
)

# =====================================================
# PARAMETERS
# =====================================================

WIN_SEC = 10.0
STEP_SEC = 2.5
K_FEATURES = 50

C_GRID = [0.1]
GAMMA_GRID = ["scale"]

THRESHOLDS = [0.0, 0.5, 1.0]
K_VALUES = [2, 4, 6, 8]

ann = pd.read_csv("seizure_annotations.csv")

# =====================================================
# STRUCTURE
# =====================================================

@dataclass
class RecordPrediction:
    record_id: str
    window_starts: np.ndarray
    window_len: float
    scores: np.ndarray
    duration_sec: float
    true_seizures: list

# =====================================================
# HELPERS (unchanged)
# =====================================================

def _runs_of_true(mask):
    runs = []
    i = 0
    while i < len(mask):
        if mask[i]:
            j = i
            while j + 1 < len(mask) and mask[j + 1]:
                j += 1
            runs.append((i, j))
            i = j + 1
        else:
            i += 1
    return runs


def interval_overlap(a_start, a_end, b_start, b_end):
    return max(0, min(a_end, b_end) - max(a_start, b_start))

# =====================================================
# EVENT DETECTION (unchanged)
# =====================================================

def detect_events(rec, threshold=0.5, k_consecutive=2, refractory_sec=120):

    pred = rec.scores >= threshold
    events = []

    for s, e in _runs_of_true(pred):
        if (e - s + 1) >= k_consecutive:
            events.append({
                "onset_sec": rec.window_starts[s],
                "offset_sec": rec.window_starts[e] + rec.window_len
            })

    events.sort(key=lambda x: x["onset_sec"])

    accepted = []
    last_t = -np.inf

    for ev in events:
        if ev["onset_sec"] >= last_t + refractory_sec:
            accepted.append(ev)
            last_t = ev["onset_sec"]

    return accepted

# =====================================================
# EVENT SCORING (unchanged)
# =====================================================

def score_one_file(rec,
                   threshold=0.5,
                   k_consecutive=2,
                   refractory_sec=120,
                   min_overlap_sec=5):

    events = detect_events(rec, threshold, k_consecutive, refractory_sec)

    matched_event = [False] * len(events)

    total_seizures = len(rec.true_seizures)
    detected_seizures = 0
    latencies = []
    seizure_hit_flags = []

    for sz_start, sz_end in rec.true_seizures:

        detected = False
        best_latency = None

        for i, ev in enumerate(events):

            overlap = interval_overlap(
                ev["onset_sec"], ev["offset_sec"],
                sz_start, sz_end
            )

            if overlap >= min_overlap_sec:
                matched_event[i] = True
                detected = True

                lat = ev["onset_sec"] - sz_start
                if best_latency is None or lat < best_latency:
                    best_latency = lat

        seizure_hit_flags.append(detected)

        if detected:
            detected_seizures += 1
            latencies.append(best_latency)

    false_detections = sum(1 for m in matched_event if not m)

    return {
        "record_id": rec.record_id,
        "events": events,
        "seizure_flags": seizure_hit_flags,
        "n_seizures": total_seizures,
        "n_detected": detected_seizures,
        "sensitivity": detected_seizures / total_seizures if total_seizures else np.nan,
        "fd_per_hour": false_detections / (rec.duration_sec / 3600),
        "median_latency": np.median(latencies) if len(latencies) else np.nan
    }

# =====================================================
# SVM TUNING (NOW RECORD-LEVEL VALIDATION)
# =====================================================

def tune_svm_record_level(X, y, groups):

    best_auc = -np.inf
    best_params = (0.1, "scale")

    gkf = GroupKFold(n_splits=5)

    for C in C_GRID:
        for gamma in GAMMA_GRID:

            fold_aucs = []

            for tr_idx, val_idx in gkf.split(X, y, groups):

                X_tr, X_val = X[tr_idx], X[val_idx]
                y_tr, y_val = y[tr_idx], y[val_idx]

                model = SVC(
                    C=C,
                    gamma=gamma,
                    kernel="rbf",
                    class_weight="balanced"
                )

                model.fit(X_tr, y_tr)
                scores = model.decision_function(X_val)

                fold_aucs.append(roc_auc_score(y_val, scores))

            mean_auc = np.mean(fold_aucs)

            if mean_auc > best_auc:
                best_auc = mean_auc
                best_params = (C, gamma)

    return best_params

# =====================================================
# LOO PIPELINE (UNCHANGED OUTER LOOP)
# =====================================================

results = []
event_results = []

for test_record in records:

    print(f"\nTesting: {test_record}")

    # ---------------- test split ----------------
    X_test, y_test = records[test_record]

    train_records = [r for r in records if r != test_record]

    X_train_all = np.vstack([records[r][0] for r in train_records])
    y_train_all = np.concatenate([records[r][1] for r in train_records])

    # =====================================================
    # GROUP LABELS (KEY FIX)
    # =====================================================

    groups = np.concatenate([
        np.full(len(records[r][1]), i)
        for i, r in enumerate(train_records)
    ])

    # =====================================================
    # FEATURE SELECTION (train-only inside tuning loop)
    # =====================================================

    selector = SelectKBest(f_classif, k=K_FEATURES)
    X_train_all = selector.fit_transform(X_train_all, y_train_all)
    X_test_fs = selector.transform(X_test)

    # =====================================================
    # SCALING (train-only distribution)
    # =====================================================

    scaler = StandardScaler()
    X_train_all = scaler.fit_transform(X_train_all)
    X_test_fs = scaler.transform(X_test_fs)

    # =====================================================
    # SVM TUNING (record-level CV)
    # =====================================================

    best_C, best_gamma = tune_svm_record_level(X_train_all, y_train_all, groups)
    print(f"Best SVM: C={best_C}, gamma={best_gamma}")

    # =====================================================
    # FINAL MODEL
    # =====================================================

    model = SVC(
        C=best_C,
        gamma=best_gamma,
        kernel="rbf",
        class_weight="balanced"
    )

    model.fit(X_train_all, y_train_all)

    y_pred = model.predict(X_test_fs)
    scores = model.decision_function(X_test_fs)

    # =====================================================
    # SEGMENT METRICS (UNCHANGED OUTPUT)
    # =====================================================

    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, scores)
    auprc = average_precision_score(y_test, scores)

    precision = precision_score(y_test, y_pred, zero_division=0)
    sensitivity = recall_score(y_test, y_pred, zero_division=0)

    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
    specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan

    print(
        f"ACC={acc:.3f} AUC={auc:.3f} AUPRC={auprc:.3f} "
        f"Sens={sensitivity:.3f} Spec={specificity:.3f} Prec={precision:.3f}"
    )

    results.append({
        "acc": acc,
        "auc": auc,
        "auprc": auprc,
        "sensitivity": sensitivity,
        "specificity": specificity,
        "precision": precision
    })

    # =====================================================
    # EVENT PIPELINE (UNCHANGED)
    # =====================================================

    window_starts = np.arange(len(y_test)) * STEP_SEC
    duration_sec = window_starts[-1] + WIN_SEC

    true_seizures = get_seizures_for_record(ann, test_record)

    rec = RecordPrediction(
        record_id=test_record,
        window_starts=window_starts,
        window_len=WIN_SEC,
        scores=scores,
        duration_sec=duration_sec,
        true_seizures=true_seizures
    )

    def tune_event_params(rec):
        best_score = -np.inf
        best = (0.5, 2)

        for th in THRESHOLDS:
            for k in K_VALUES:
                out = score_one_file(rec, threshold=th, k_consecutive=k)
                score = out["sensitivity"] - 0.01 * out["fd_per_hour"]

                if score > best_score:
                    best_score = score
                    best = (th, k)

        return best

    best_th, best_k = tune_event_params(rec)

    print(f"Best event params: th={best_th}, k={best_k}")

    out = score_one_file(rec, threshold=best_th, k_consecutive=best_k)

    event_results.append(out)

    print("Seizures:", out["n_seizures"])
    print("Detected:", out["n_detected"])
    print("Sensitivity:", out["sensitivity"])
    print("FD/hour:", out["fd_per_hour"])
    print("Median latency:", out["median_latency"])

# =====================================================
# FINAL SUMMARY (UNCHANGED)
# =====================================================

print("\n================ SEGMENT SUMMARY ================")

print("Mean ACC :", np.mean([r["acc"] for r in results]))
print("Mean AUC :", np.mean([r["auc"] for r in results]))
print("Mean AUPRC:", np.mean([r["auprc"] for r in results]))
print("Mean Sens:", np.mean([r["sensitivity"] for r in results]))
print("Mean Spec:", np.mean([r["specificity"] for r in results]))
print("Mean Prec:", np.mean([r["precision"] for r in results]))

print("\n================ EVENT SUMMARY ================")

print("Mean Event Sens:", np.nanmean([r["sensitivity"] for r in event_results]))
print("Mean FD/hour   :", np.nanmean([r["fd_per_hour"] for r in event_results]))

In [ ]:
"""
Leave-one-record-out (LORO) seizure-detection pipeline.

Expects from your environment (same as before):
  - records: dict[str, (X, y)]  feature matrix + window labels per record
  - get_seizures_for_record(ann, record_id) -> list[(onset_sec, offset_sec)]
  - seizure_annotations.csv

What changed vs. your version (the important bits):
  1. EVENT PARAMS ARE NO LONGER TUNED ON THE TEST RECORD.
     Threshold is calibrated on the TRAINING records using OUT-OF-FOLD scores
     to hit a target FD/hour budget; k is fixed a priori (Shoeb-aligned).
  2. Preprocessing (SelectKBest + StandardScaler) lives inside an sklearn
     Pipeline, so it is refit per fold during tuning -> no inner-CV leakage.
  3. Single-class guards on ROC-AUC / AUPRC so seizure-free test records
     (or single-class val folds) don't crash the run.
"""

from dataclasses import dataclass
import numpy as np
import pandas as pd

from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    average_precision_score,
    recall_score,
    precision_score,
    confusion_matrix,
)

# =====================================================
# PARAMETERS
# =====================================================

WIN_SEC = 10.0
STEP_SEC = 2.5
K_FEATURES = 50

C_GRID = [0.1]
GAMMA_GRID = ["scale"]

# Event-level operating point (chosen on TRAIN, applied to TEST)
TARGET_FD_PER_HOUR = 0.5          # false-alarm budget
K_FIXED = 4                        # persistence, fixed a priori (Shoeb-aligned)
REFRACTORY_SEC = 120
MIN_OVERLAP_SEC = 5
THRESHOLD_GRID = np.linspace(-2.0, 2.0, 41)   # decision_function space

N_SPLITS = 5                       # capped per training-set size below

ann = pd.read_csv("seizure_annotations.csv")

# =====================================================
# STRUCTURE
# =====================================================

@dataclass
class RecordPrediction:
    record_id: str
    window_starts: np.ndarray
    window_len: float
    scores: np.ndarray
    duration_sec: float
    true_seizures: list

# =====================================================
# HELPERS
# =====================================================

def _runs_of_true(mask):
    runs = []
    i = 0
    while i < len(mask):
        if mask[i]:
            j = i
            while j + 1 < len(mask) and mask[j + 1]:
                j += 1
            runs.append((i, j))
            i = j + 1
        else:
            i += 1
    return runs


def interval_overlap(a_start, a_end, b_start, b_end):
    return max(0, min(a_end, b_end) - max(a_start, b_start))


def safe_auc(y_true, scores):
    return roc_auc_score(y_true, scores) if len(np.unique(y_true)) > 1 else np.nan


def safe_auprc(y_true, scores):
    return average_precision_score(y_true, scores) if len(np.unique(y_true)) > 1 else np.nan


def build_record_prediction(record_id, scores):
    n = len(scores)
    window_starts = np.arange(n) * STEP_SEC
    duration_sec = (window_starts[-1] + WIN_SEC) if n else 0.0
    return RecordPrediction(
        record_id=record_id,
        window_starts=window_starts,
        window_len=WIN_SEC,
        scores=np.asarray(scores),
        duration_sec=duration_sec,
        true_seizures=get_seizures_for_record(ann, record_id),
    )

# =====================================================
# EVENT DETECTION
# =====================================================

def detect_events(rec, threshold, k_consecutive, refractory_sec=REFRACTORY_SEC):
    pred = rec.scores >= threshold
    events = []

    for s, e in _runs_of_true(pred):
        if (e - s + 1) >= k_consecutive:
            events.append({
                "onset_sec": rec.window_starts[s],
                "offset_sec": rec.window_starts[e] + rec.window_len,
            })

    events.sort(key=lambda x: x["onset_sec"])

    accepted = []
    last_t = -np.inf
    for ev in events:
        if ev["onset_sec"] >= last_t + refractory_sec:
            accepted.append(ev)
            last_t = ev["onset_sec"]

    return accepted

# =====================================================
# EVENT SCORING
# =====================================================

def score_one_file(rec, threshold, k_consecutive,
                   refractory_sec=REFRACTORY_SEC, min_overlap_sec=MIN_OVERLAP_SEC):

    events = detect_events(rec, threshold, k_consecutive, refractory_sec)
    matched_event = [False] * len(events)

    total_seizures = len(rec.true_seizures)
    detected_seizures = 0
    latencies = []
    seizure_hit_flags = []

    for sz_start, sz_end in rec.true_seizures:
        detected = False
        best_latency = None

        for i, ev in enumerate(events):
            overlap = interval_overlap(
                ev["onset_sec"], ev["offset_sec"], sz_start, sz_end
            )
            if overlap >= min_overlap_sec:
                matched_event[i] = True
                detected = True
                # latency relative to seizure onset, clamped at 0
                lat = max(0.0, ev["onset_sec"] - sz_start)
                if best_latency is None or lat < best_latency:
                    best_latency = lat

        seizure_hit_flags.append(detected)
        if detected:
            detected_seizures += 1
            latencies.append(best_latency)

    false_detections = sum(1 for m in matched_event if not m)

    return {
        "record_id": rec.record_id,
        "events": events,
        "seizure_flags": seizure_hit_flags,
        "n_seizures": total_seizures,
        "n_detected": detected_seizures,
        "n_false": false_detections,          # raw count, for pooling
        "duration_hours": rec.duration_sec / 3600,
        "sensitivity": detected_seizures / total_seizures if total_seizures else np.nan,
        "fd_per_hour": false_detections / (rec.duration_sec / 3600) if rec.duration_sec else np.nan,
        "median_latency": np.median(latencies) if len(latencies) else np.nan,
    }

# =====================================================
# THRESHOLD CALIBRATION (ON TRAINING RECORDS)
# =====================================================

def choose_threshold_for_fd_target(train_recs, target_fd_per_hour, k_consecutive,
                                    candidate_thresholds=THRESHOLD_GRID):
    """Lowest threshold whose POOLED train FD/hour <= target.
    Lowest qualifying threshold = max sensitivity within the false-alarm budget."""
    best_th = float(np.max(candidate_thresholds))   # most conservative fallback
    for th in np.sort(candidate_thresholds):        # low -> high
        total_false, total_hours = 0, 0.0
        for rec in train_recs:
            out = score_one_file(rec, threshold=th, k_consecutive=k_consecutive)
            total_false += out["n_false"]
            total_hours += out["duration_hours"]
        pooled_fd = total_false / total_hours if total_hours else np.inf
        if pooled_fd <= target_fd_per_hour:
            best_th = float(th)
            break
    return best_th

# =====================================================
# SVM TUNING (record-level CV, leak-free pipeline)
# =====================================================

def make_pipeline(C, gamma):
    return Pipeline([
        ("select", SelectKBest(f_classif, k=K_FEATURES)),
        ("scale", StandardScaler()),
        ("svc", SVC(C=C, gamma=gamma, kernel="rbf", class_weight="balanced")),
    ])


def tune_svm_record_level(X, y, groups, n_splits):
    best_auc = -np.inf
    best_params = (C_GRID[0], GAMMA_GRID[0])
    gkf = GroupKFold(n_splits=n_splits)

    for C in C_GRID:
        for gamma in GAMMA_GRID:
            fold_aucs = []
            for tr_idx, val_idx in gkf.split(X, y, groups):
                pipe = make_pipeline(C, gamma)
                pipe.fit(X[tr_idx], y[tr_idx])           # preprocessing refit per fold
                scores = pipe.decision_function(X[val_idx])
                fold_aucs.append(safe_auc(y[val_idx], scores))

            mean_auc = np.nanmean(fold_aucs)
            if mean_auc > best_auc:
                best_auc = mean_auc
                best_params = (C, gamma)

    return best_params

# =====================================================
# LORO PIPELINE
# =====================================================

results = []
event_results = []

for test_record in records:

    print(f"\nTesting: {test_record}")

    X_test, y_test = records[test_record]
    train_records = [r for r in records if r != test_record]

    # record-ordered training matrix + boundaries (to split OOF scores back per record)
    X_train_all = np.vstack([records[r][0] for r in train_records])
    y_train_all = np.concatenate([records[r][1] for r in train_records])

    lengths = [len(records[r][1]) for r in train_records]
    bounds = np.cumsum([0] + lengths)

    groups = np.concatenate([
        np.full(lengths[i], i) for i in range(len(train_records))
    ])

    n_splits = max(2, min(N_SPLITS, len(train_records)))
    gkf = GroupKFold(n_splits=n_splits)

    # ---------------- SVM tuning (leak-free) ----------------
    best_C, best_gamma = tune_svm_record_level(X_train_all, y_train_all, groups, n_splits)
    print(f"Best SVM: C={best_C}, gamma={best_gamma}")

    pipe = make_pipeline(best_C, best_gamma)

    # ---------------- OUT-OF-FOLD train scores -> threshold calibration ----------------
    oof_scores = cross_val_predict(
        pipe, X_train_all, y_train_all,
        groups=groups, cv=gkf, method="decision_function",
    )
    train_recs = [
        build_record_prediction(r, oof_scores[bounds[i]:bounds[i + 1]])
        for i, r in enumerate(train_records)
    ]
    best_th = choose_threshold_for_fd_target(
        train_recs, target_fd_per_hour=TARGET_FD_PER_HOUR, k_consecutive=K_FIXED
    )
    print(f"Calibrated event threshold (train): th={best_th:.3f}, k={K_FIXED}")

    # ---------------- final model -> test record ----------------
    pipe.fit(X_train_all, y_train_all)
    scores = pipe.decision_function(X_test)
    y_pred = (scores >= 0).astype(int)

    # ---------------- segment metrics (guarded) ----------------
    acc = accuracy_score(y_test, y_pred)
    auc = safe_auc(y_test, scores)
    auprc = safe_auprc(y_test, scores)
    precision = precision_score(y_test, y_pred, zero_division=0)
    sensitivity = recall_score(y_test, y_pred, zero_division=0)

    tn, fp, fn, tp = confusion_matrix(y_test, y_pred, labels=[0, 1]).ravel()
    specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan

    print(
        f"ACC={acc:.3f} AUC={auc:.3f} AUPRC={auprc:.3f} "
        f"Sens={sensitivity:.3f} Spec={specificity:.3f} Prec={precision:.3f}"
    )

    results.append({
        "acc": acc, "auc": auc, "auprc": auprc,
        "sensitivity": sensitivity, "specificity": specificity, "precision": precision,
    })

    # ---------------- event metrics (frozen train-calibrated params) ----------------
    rec = build_record_prediction(test_record, scores)
    out = score_one_file(rec, threshold=best_th, k_consecutive=K_FIXED)
    event_results.append(out)

    print("Seizures      :", out["n_seizures"])
    print("Detected      :", out["n_detected"])
    print("Sensitivity   :", out["sensitivity"])
    print("FD/hour       :", out["fd_per_hour"])
    print("Median latency:", out["median_latency"])

# =====================================================
# FINAL SUMMARY
# =====================================================

print("\n================ SEGMENT SUMMARY ================")
print("Mean ACC  :", np.nanmean([r["acc"] for r in results]))
print("Mean AUC  :", np.nanmean([r["auc"] for r in results]))
print("Mean AUPRC:", np.nanmean([r["auprc"] for r in results]))
print("Mean Sens :", np.nanmean([r["sensitivity"] for r in results]))
print("Mean Spec :", np.nanmean([r["specificity"] for r in results]))
print("Mean Prec :", np.nanmean([r["precision"] for r in results]))

print("\n================ EVENT SUMMARY ================")
# sensitivity pooled over seizure records; FD/hour pooled over ALL records
total_detected = np.nansum([r["n_detected"] for r in event_results])
total_seizures = np.nansum([r["n_seizures"] for r in event_results])
total_false = np.nansum([r["n_false"] for r in event_results])
total_hours = np.nansum([r["duration_hours"] for r in event_results])

print("Event Sensitivity (pooled):", total_detected / total_seizures if total_seizures else np.nan)
print("FD/hour (pooled)          :", total_false / total_hours if total_hours else np.nan)
print("Median latency            :", np.nanmedian([r["median_latency"] for r in event_results]))


Testing: chb01_03
Best SVM: C=0.1, gamma=scale
Calibrated event threshold (train): th=-0.300, k=4
ACC=0.990 AUC=0.977 AUPRC=0.418 Sens=0.438 Spec=0.996 Prec=0.538
Seizures      : 1
Detected      : 1
Sensitivity   : 1.0
FD/hour       : 2.0
Median latency: 19.0

Testing: chb01_04
Best SVM: C=0.1, gamma=scale
Calibrated event threshold (train): th=-0.300, k=4
ACC=0.985 AUC=0.990 AUPRC=0.284 Sens=0.727 Spec=0.987 Prec=0.296
Seizures      : 1
Detected      : 1
Sensitivity   : 1.0
FD/hour       : 1.0
Median latency: 0.0

Testing: chb01_15
Best SVM: C=0.1, gamma=scale
Calibrated event threshold (train): th=-0.300, k=4
ACC=0.962 AUC=0.790 AUPRC=0.032 Sens=0.062 Spec=0.972 Prec=0.024
Seizures      : 1
Detected      : 1
Sensitivity   : 1.0
FD/hour       : 3.0
Median latency: 33.0

Testing: chb01_16
Best SVM: C=0.1, gamma=scale
Calibrated event threshold (train): th=-0.200, k=4
ACC=0.980 AUC=0.981 AUPRC=0.319 Sens=0.381 Spec=0.989 Prec=0.333
Seizures      : 1
Detected      : 1
Sensitivity   : 1.